<a href="https://colab.research.google.com/github/Hamdi-Jarban/-/blob/main/thai_craft_finetune_v3_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import importlib
import subprocess
import sys

_REQUIRED = ["pythainlp", "PIL", "numpy", "cv2", "skimage", "matplotlib"]
_PIP_NAMES = {
    "PIL": "pillow",
    "cv2": "opencv-python",
    "skimage": "scikit-image",
}

_missing = []
for _mod in _REQUIRED:
    try:
        importlib.import_module(_mod)
    except ImportError:
        _missing.append(_PIP_NAMES.get(_mod, _mod))

if _missing:
    print("تثبيت الحزم الناقصة:", _missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
else:
    print("كل المكتبات المطلوبة مثبَّتة بالفعل.")

تثبيت الحزم الناقصة: ['pythainlp']


In [ ]:

try:
    from google.colab import drive
    drive.mount('/content/drive')
    _ON_COLAB = True
except ImportError:
    _ON_COLAB = False
    print("لسنا داخل Google Colab — تخطّي ربط Drive (تأكد من ضبط FONT_DIR/CHECKPOINT_DIR يدوياً أدناه).")


Mounted at /content/drive


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive') if _ON_COLAB else Path('./thai_craft_local')
FONT_DIR = DRIVE_ROOT / 'thai_assets' / 'fonts' / 'thai'
CHECKPOINT_DIR = DRIVE_ROOT / 'thai_craft_checkpoints'
PREVIEW_DIR = CHECKPOINT_DIR / 'online_previews'

# جديد: أوزان CRAFT الرسمية المُدرَّبة مسبقاً (نقطة انطلاق التخصيص بدلاً من العشوائية الكاملة)
PRETRAINED_DIR = DRIVE_ROOT / 'thai_assets' / 'pretrained'
PRETRAINED_CRAFT_PATH =  Path('/content/drive/MyDrive/thai_craft_checkpoints/online_previews/craft.pth')

# جديد: خلفيات حقيقية اختيارية (لن تُنشئ الخلية التالية أي خطأ إن تركته فارغاً)
BACKGROUND_DIR = DRIVE_ROOT / 'thai_assets' / 'backgrounds'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
BACKGROUND_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = PREVIEW_DIR / 'craft_latest.pt'
BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / 'craft_online_best.pt'

assert FONT_DIR.exists(), (
    f"مجلد الخطوط غير موجود: {FONT_DIR}\n"
    "ضع بداخله خطوطاً تايلاندية بصيغة .ttf أو .otf (Sarabun, Kanit, Prompt, Mitr, Noto Sans Thai ...)."
)
_font_files = list(FONT_DIR.rglob('*.ttf')) + list(FONT_DIR.rglob('*.otf'))
assert len(_font_files) > 0, f"لا توجد أي خطوط (.ttf/.otf) داخل: {FONT_DIR}"

_bg_files = (list(BACKGROUND_DIR.rglob('*.jpg')) + list(BACKGROUND_DIR.rglob('*.jpeg'))
             + list(BACKGROUND_DIR.rglob('*.png')) + list(BACKGROUND_DIR.rglob('*.webp')))

print("عدد الخطوط المتاحة:", len(_font_files))
print("عدد صور الخلفيات الحقيقية المتاحة:", len(_bg_files),
      "(فارغ = سيُعتمَد على الخلفيات الإجرائية فقط، وهذا آمن تماماً)")
print("Checkpoint الأخير موجود مسبقاً:", CHECKPOINT_PATH.exists())
print("أفضل Checkpoint موجود مسبقاً:", BEST_CHECKPOINT_PATH.exists())
print("أفضل Checkpoint موجود مسبقاً:", PRETRAINED_CRAFT_PATH.exists())



عدد الخطوط المتاحة: 96
عدد صور الخلفيات الحقيقية المتاحة: 685 (فارغ = سيُعتمَد على الخلفيات الإجرائية فقط، وهذا آمن تماماً)
Checkpoint الأخير موجود مسبقاً: True
أفضل Checkpoint موجود مسبقاً: True
أفضل Checkpoint موجود مسبقاً: True


In [ ]:
# تنزيل أوزان CRAFT الرسمية المُدرَّبة مسبقاً (craft_mlt_25k.pth) إن لم تكن موجودة بعد.
# المصدر الرسمي: مستودع clovaai/CRAFT-pytorch (Naver Clova AI) — نموذج "General" المدرَّب على
# SynthText + ICDAR13 + ICDAR17 (إنجليزي + متعدد اللغات). هذا هو المُستخدم كنقطة انطلاق للتخصيص أدناه.
_CRAFT_MLT_25K_GDRIVE_ID = "1Jk4eGD7crsqCCg9C9VjCLkMN3ze8kutZ"

if PRETRAINED_CRAFT_PATH.exists() and PRETRAINED_CRAFT_PATH.stat().st_size > 1_000_000:
    print("الأوزان المُدرَّبة مسبقاً موجودة بالفعل:", PRETRAINED_CRAFT_PATH)
else:
    try:
        import importlib.util, subprocess, sys as _sys
        if importlib.util.find_spec("gdown") is None:
            subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
        url = f"https://drive.google.com/uc?id={_CRAFT_MLT_25K_GDRIVE_ID}"
        gdown.download(url, str(PRETRAINED_CRAFT_PATH), quiet=False)
        assert PRETRAINED_CRAFT_PATH.exists() and PRETRAINED_CRAFT_PATH.stat().st_size > 1_000_000, \
            "الملف الناتج صغير جداً أو غير موجود — التنزيل التلقائي فشل."
        print("تم تنزيل الأوزان المُدرَّبة مسبقاً بنجاح:", PRETRAINED_CRAFT_PATH)
    except Exception as e:
        print("[تنبيه] فشل التنزيل التلقائي للأوزان المُدرَّبة مسبقاً. السبب:", repr(e))
        print("الحل: نزّل الملف يدوياً من الرابط التالي وارفعه إلى المسار المذكور:")
        print(f"  https://drive.google.com/uc?id={_CRAFT_MLT_25K_GDRIVE_ID}")
        print(f"  المسار المطلوب: {PRETRAINED_CRAFT_PATH}")
        print("سيتابع الدفتر العمل حتى بدون هذا الملف، لكنه سيبدأ من أوزان عشوائية (كما في السابق) بدلاً من التخصيص.")


الأوزان المُدرَّبة مسبقاً موجودة بالفعل: /content/drive/MyDrive/thai_craft_checkpoints/online_previews/craft.pth


## 2) مولّد البيانات الاصطناعي (في الذاكرة أساساً، + خلفيات حقيقية اختيارية من Drive)

يشمل:
1. تقسيم النص التايلاندي إلى عناقيد حروف (Grapheme Clustering).
2. `CorpusGenerator`: كلمات/جمل/أرقام من قاموس pythainlp الضخم، **+ حَقن صريح للحروف الأثرية/المهملة**
   (ฃ, ฅ, ฤ, ฦ, ฤๅ, ฦๅ) لأنها شبه غائبة عن أي نص حديث طبيعياً.
3. `FontManager`: تحميل الخطوط من `FONT_DIR`.
4. `BackgroundGenerator`: خلفيات إجرائية (ورق/شارع/غرفة/طبيعة/وثيقة عتيقة) + مزج اختياري مع صور حقيقية من
   `BACKGROUND_DIR` إن وُجدت.
5. عرض النص وحساب صندوق دقيق لكل حرف (Ink-tight polygon).
6. تمويهات/ضوضاء بشدة قابلة للتدرّج.
7. تركيب مشهد كامل (خلفية + عدّة كتل نص) بلا تداخل.


In [ ]:
import os
import re
import glob
import math
import random
import unicodedata
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance, features

if not features.check("raqm"):
    print("[تنبيه] Pillow لا يدعم raqm — تكديس العلامات فوق/تحت الحرف الأساسي قد لا يكون دقيقاً تماماً.")

Point = Tuple[float, float]
Polygon = List[Point]

# --------------------------------------------------------------------------
# تقسيم النص التايلاندي إلى عناقيد حروف (حرف أساسي + علاماته المرتبطة به)
# --------------------------------------------------------------------------
_THAI_COMBINING = (
    "\u0E31"
    "\u0E34\u0E35\u0E36\u0E37\u0E38\u0E39\u0E3A"
    "\u0E47\u0E48\u0E49\u0E4A\u0E4B\u0E4C\u0E4D\u0E4E"
)
_CLUSTER_RE = re.compile(r"[^\s]" + f"[{_THAI_COMBINING}]*" + r"|\s", re.UNICODE)


def split_graphemes(text: str) -> List[str]:
    '''يقسّم نصاً تايلاندياً إلى عناقيد (كل عنقود = حرف أساسي + علاماته).'''
    return [m.group(0) for m in _CLUSTER_RE.finditer(text) if m.group(0) != ""]


try:
    from pythainlp.corpus import thai_words
    from pythainlp.tokenize import sent_tokenize, word_tokenize
    from pythainlp.util import normalize as thai_normalize
    _PYTHAINLP_AVAILABLE = True
except ImportError:
    _PYTHAINLP_AVAILABLE = False
    def thai_normalize(text: str) -> str:
        return unicodedata.normalize("NFC", text)

_FALLBACK_WORDS = [
    "สวัสดี", "ประเทศไทย", "ขอบคุณ", "อาหาร", "โรงเรียน", "รถยนต์", "แมว", "หมา",
    "น้ำ", "ไฟฟ้า", "ต้นไม้", "ดอกไม้", "ท้องฟ้า", "ทะเล", "ภูเขา", "หนังสือ",
]
_FALLBACK_SENTENCES = [
    "วันนี้อากาศดีมาก เหมาะแก่การเดินทางท่องเที่ยว",
    "กรุณาต่อแถวเพื่อรอรับบริการอย่างเป็นระเบียบ",
    "ยินดีต้อนรับสู่ร้านอาหารของเรา",
]

_MIN_WORD_LEN, _MAX_WORD_LEN = 1, 15
_MIN_SENT_LEN, _MAX_SENT_LEN = 8, 60

_DEFAULT_CORPUS_WEIGHTS = {"word": 35, "sentence": 30, "digits": 15, "multiline": 20}


def _random_number_string() -> str:
    kind = random.choice(["plain", "phone", "money", "date", "percent"])
    if kind == "plain":
        return str(random.randint(0, 999999))
    if kind == "phone":
        return f"0{random.randint(60, 99)}-{random.randint(100, 999)}-{random.randint(1000, 9999)}"
    if kind == "money":
        return f"{random.randint(1, 99999):,}.{random.randint(0, 99):02d} บาท"
    if kind == "date":
        return f"{random.randint(1, 31):02d}/{random.randint(1, 12):02d}/{random.randint(2560, 2568)}"
    return f"{random.randint(0, 100)}%"


class CorpusGenerator:
    '''يولّد عناصر نصية عشوائية (كلمة/جملة/رقم/عدة أسطر) من قاموس pythainlp
    الضخم. يدعم أوزاناً ديناميكية عبر sample(weights=...) لأجل التعلّم التدريجي.'''

    def __init__(self, corpus_file: Optional[str] = None):
        self.words: List[str] = []
        self.sentences: List[str] = []

        if _PYTHAINLP_AVAILABLE:
            all_words = list(thai_words())
            self.words = [w for w in all_words if _MIN_WORD_LEN <= len(w) <= _MAX_WORD_LEN]
            print(f"[CorpusGenerator] تم تحميل {len(self.words)} كلمة من pythainlp.corpus.thai_words()")
        else:
            print("[تنبيه] pythainlp غير مثبّتة — سيُستخدم قاموس احتياطي صغير جداً "
                  "(غير كافٍ لتدريب جاد؛ ثبّت pythainlp للحصول على تنوّع حقيقي).")
            self.words = list(_FALLBACK_WORDS)
            self.sentences = list(_FALLBACK_SENTENCES)

        if corpus_file and os.path.isfile(corpus_file):
            with open(corpus_file, "r", encoding="utf-8") as f:
                raw_text = f.read()
            if _PYTHAINLP_AVAILABLE:
                extracted_sents = [s.strip() for s in sent_tokenize(raw_text) if s.strip()]
                self.sentences.extend(s for s in extracted_sents if _MIN_SENT_LEN <= len(s) <= _MAX_SENT_LEN)
                extracted_words = [w.strip() for w in word_tokenize(raw_text) if w.strip()]
                self.words.extend(w for w in extracted_words if _MIN_WORD_LEN <= len(w) <= _MAX_WORD_LEN)
            else:
                for line in raw_text.splitlines():
                    line = line.strip()
                    if not line:
                        continue
                    (self.sentences if " " in line or len(line) > 12 else self.words).append(line)

        self.words = list(dict.fromkeys(self.words)) or list(_FALLBACK_WORDS)
        self.sentences = list(dict.fromkeys(self.sentences))
        self._synthesize_phrases = len(self.sentences) < 50

    def _random_phrase(self) -> str:
        n = random.randint(2, 6)
        return " ".join(random.choice(self.words) for _ in range(n))

    def sample(self, weights: Optional[dict] = None) -> Tuple[str, str]:
        '''يرجع (نص مُطبَّع, نوع). weights اختياري لتحكّم مستوى الصعوبة،
        مثال: {'word': 70, 'sentence': 10, 'digits': 15, 'multiline': 5}'''
        w = weights or _DEFAULT_CORPUS_WEIGHTS
        kinds = ["word", "sentence", "digits", "multiline"]
        kind = random.choices(kinds, weights=[w.get(k, 1) for k in kinds])[0]

        if kind == "word":
            text = random.choice(self.words)
        elif kind == "digits":
            text = _random_number_string()
        elif kind == "sentence":
            if self.sentences and not (self._synthesize_phrases and random.random() < 0.5):
                text = random.choice(self.sentences)
            else:
                text = self._random_phrase()
        else:  # multiline
            n = random.randint(2, 4)
            lines = []
            for _ in range(n):
                if self.sentences and random.random() < 0.4:
                    lines.append(random.choice(self.sentences))
                else:
                    lines.append(random.choice(self.words))
            text = "\n".join(lines)

        return thai_normalize(text), kind

    def inject_archaic(self, text: str) -> str:
        '''يحوّل نصاً عادياً إلى نسخة تحتوي حرفاً أثرياً/مهملاً، حتى يرى النموذج هذه الحروف
        ضمن سياق طبيعي شبيه بالنصوص/النقوش الحقيقية، وليس فقط بمعزل تام. ثلاث طرق:
        (أ) حرف أثري منعزل تماماً، (ب) استبدال تهجئة أثري موثّق تاريخياً داخل كلمة حقيقية
        (ฃ بدل ข، ฅ بدل ค — كما في نقش رามคำแหง)، (ج) كلمة تايلاندية حديثة حقيقية لا تزال
        تستخدم ฤ (ليست مهملة، لكنها نادرة الظهور في عيّنة عشوائية من القاموس).'''
        roll = random.random()
        if roll < 0.35:
            return random.choice(_ARCHAIC_STANDALONE)
        substitutable = [ch for ch in text if ch in _ARCHAIC_SUBSTITUTIONS]
        if roll < 0.60 and substitutable:
            chars = list(text)
            idxs = [k for k, c in enumerate(chars) if c in _ARCHAIC_SUBSTITUTIONS]
            k = random.choice(idxs)
            chars[k] = _ARCHAIC_SUBSTITUTIONS[chars[k]]
            return "".join(chars)
        return random.choice(_ARCHAIC_REAL_WORDS)


# تهجئات أثرية موثّقة تاريخياً: ฃ (كو-خวด) و ฅ (คو-คน) كانتا حرفين منفصلين صوتياً، ثم اندمجتا
# لاحقاً في النطق مع ข وค على التوالي وأصبحتا "مهملتين" رسمياً — لكنهما تظهران في النقوش والوثائق
# القديمة (مثل نقش رามคำแหง الشهير) بدل ข/ค في نفس الكلمات تماماً.
_ARCHAIC_SUBSTITUTIONS = {"ข": "ฃ", "ค": "ฅ"}
# حروف أثرية/نادرة الاستخدام لا تقبل استبدالاً مباشراً في كلمات حديثة (تُستخدم منعزلة أو ضمن ฤๅ/ฦๅ)
_ARCHAIC_STANDALONE = ["ฃ", "ฅ", "ฤ", "ฦ", "ฤๅ", "ฦๅ"]
# كلمات تايلاندية حديثة حقيقية (ليست أثرية) لكنها من الأمثلة النادرة التي ما زالت تحوي ฤ في الاستخدام اليومي
_ARCHAIC_REAL_WORDS = ["ฤดู", "ฤดูกาล", "พฤษภาคม", "ฤทธิ์", "ฤๅษี", "ทฤษฎี", "หฤทัย", "พฤหัสบดี"]

print("تم تعريف: split_graphemes, CorpusGenerator (+ حَقن الحروف الأثرية عبر inject_archaic)")


تم تعريف: split_graphemes, CorpusGenerator (+ حَقن الحروف الأثرية عبر inject_archaic)


In [ ]:
class FontManager:
    def __init__(self, fonts_dir: str):
        self.paths = sorted(
            glob.glob(os.path.join(fonts_dir, "**", "*.ttf"), recursive=True)
            + glob.glob(os.path.join(fonts_dir, "**", "*.otf"), recursive=True)
        )
        if not self.paths:
            raise RuntimeError(
                f"لم يتم العثور على أي خط (.ttf/.otf) داخل: {fonts_dir}\n"
                "ضع فيه خطوطاً تايلاندية (Sarabun, Kanit, Prompt, Mitr, Noto Sans Thai ...)."
            )
        if len(self.paths) < 10:
            print(f"[تنبيه] تم العثور على {len(self.paths)} خط فقط (يُفضَّل 10+ للتنوع). المتابعة بما هو متاح.")

    def random_font(self, size: int) -> ImageFont.FreeTypeFont:
        path = random.choice(self.paths)
        size = max(8, int(size))
        try:
            return ImageFont.truetype(path, size=size, layout_engine=ImageFont.Layout.RAQM)
        except Exception:
            return ImageFont.truetype(path, size=size)

    def font_at(self, index: int, size: int) -> ImageFont.FreeTypeFont:
        path = self.paths[index % len(self.paths)]
        size = max(8, int(size))
        try:
            return ImageFont.truetype(path, size=size, layout_engine=ImageFont.Layout.RAQM)
        except Exception:
            return ImageFont.truetype(path, size=size)


print("تم تعريف: FontManager")


تم تعريف: FontManager


In [ ]:
class BackgroundGenerator:
    '''مولّد خلفيات إجرائي بالكامل — لا يعتمد على أي ملفات صور خارجية إطلاقاً.
    مستوى complexity في [0, 1] يتحكّم بمدى تعقيد وضوضاء الخلفية (يُستخدم في
    التعلّم التدريجي: يبدأ التدريب بخلفيات بسيطة جداً ثم يزداد التعقيد).'''

    def __init__(self, size: int, background_dir: Optional[str] = None):
        self.size = size
        self.bg_paths: List[str] = []
        if background_dir:
            bg_dir = Path(background_dir)
            if bg_dir.exists():
                exts = ("*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp")
                self.bg_paths = [str(p) for ext in exts for p in bg_dir.rglob(ext)]
        if self.bg_paths:
            print(f"[BackgroundGenerator] تم العثور على {len(self.bg_paths)} صورة خلفية حقيقية "
                  "— ستُخلط مع الخلفيات الإجرائية.")
        else:
            print("[BackgroundGenerator] لا توجد خلفيات حقيقية (أو المجلد فارغ/غير موجود) "
                  "— الاعتماد على الخلفيات الإجرائية فقط، وهذا آمن تماماً.")

    def _noise_layer(self, img: np.ndarray, strength: float) -> np.ndarray:
        if strength <= 0:
            return img
        noise = np.random.normal(0, strength, img.shape).astype(np.float32)
        return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    def _organic_texture(self, shape, scale=24, blur=0.0):
        small = np.random.randint(0, 256, (max(2, self.size // scale), max(2, self.size // scale), 1), dtype=np.uint8)
        tex = cv2.resize(small, (self.size, self.size), interpolation=cv2.INTER_CUBIC)
        if blur:
            tex = cv2.GaussianBlur(tex, (0, 0), blur)
        return tex.astype(np.float32) / 255.0

    def _flat_paper(self) -> np.ndarray:
        # أبسط خلفية ممكنة: لون شبه مصمت مع اختلاف طفيف جداً — لمرحلة البداية فقط
        base = random.randint(210, 250)
        img = np.full((self.size, self.size, 3), base, dtype=np.float32)
        img = self._noise_layer(img, random.uniform(0.5, 2.0))
        return np.clip(img, 0, 255).astype(np.uint8)

    def _paper_texture(self) -> np.ndarray:
        base = random.randint(190, 246)
        img = np.full((self.size, self.size, 3), base, dtype=np.float32)
        fine = self._organic_texture(img.shape, scale=random.randint(18, 35))
        img += (fine[..., None] - .5) * random.uniform(18, 42)
        for _ in range(random.randint(5, 18)):
            x, y = random.randrange(self.size), random.randrange(self.size)
            rx, ry = random.randint(3, self.size // 5), random.randint(2, self.size // 12)
            layer = np.zeros((self.size, self.size), np.uint8)
            cv2.ellipse(layer, (x, y), (rx, ry), random.uniform(0, 180), 0, 360, random.randint(25, 130), -1)
            layer = cv2.GaussianBlur(layer, (0, 0), random.uniform(2, 12)) / 255.0
            img -= layer[..., None] * random.uniform(8, 34)
        if random.random() < 0.55:
            for y in range(random.randint(24, 70), self.size, random.randint(35, 75)):
                cv2.line(img, (0, y), (self.size, y + random.randint(-2, 2)), (170, 170, 185), 1)
        return self._noise_layer(np.clip(img, 0, 255).astype(np.uint8), random.uniform(1.5, 6))

    def _street_sign(self) -> np.ndarray:
        h = self.size
        sky = np.linspace(np.array([75, 120, 165]), np.array([210, 190, 145]), h).astype(np.uint8)
        img = np.repeat(sky[:, None, :], self.size, axis=1)
        horizon = random.randint(int(.52 * h), int(.72 * h))
        img[horizon:] = np.array([65, 65, 58], dtype=np.uint8)
        cv2.line(img, (0, horizon), (self.size, horizon + random.randint(-8, 8)), (210, 190, 150), max(1, h // 180))
        for _ in range(random.randint(3, 8)):
            x = random.randint(-h // 4, h)
            cv2.line(img, (x, horizon), (x + random.randint(-h // 8, h // 8), 0), (35, 35, 34), max(2, h // 70))
        x, y = random.randint(h // 10, h // 2), random.randint(h // 8, h // 2)
        pw, ph = random.randint(h // 3, int(.8 * h)), random.randint(h // 8, h // 3)
        slant = random.randint(-h // 12, h // 12)
        pts = np.float32([(x, y), (x + pw, y + slant), (x + pw - 10, y + ph + slant), (x - 10, y + ph)])
        cv2.fillConvexPoly(img, pts.astype(np.int32), random.choice(
            [(22, 65, 54), (24, 56, 100), (112, 40, 34), (45, 45, 43), (176, 145, 38)]))
        cv2.polylines(img, [pts.astype(np.int32)], True, (210, 205, 180), max(2, h // 100))
        sigma = random.choice([0.0, 0.4, 1.0])
        if sigma > 0:
            img = cv2.GaussianBlur(img, (0, 0), sigma)
        return self._noise_layer(img, random.uniform(2, 8))

    def _indoor_gradient(self) -> np.ndarray:
        top = np.random.randint(105, 220, size=3)
        bottom = np.random.randint(45, 165, size=3)
        img = np.repeat(np.linspace(top, bottom, self.size).astype(np.uint8)[:, None, :], self.size, axis=1)
        y0 = random.randint(int(.55 * self.size), int(.78 * self.size))
        cv2.rectangle(img, (0, y0), (self.size, self.size), (55, 48, 42), -1)
        if random.random() < .75:
            wx = random.randint(0, self.size // 2)
            ww = random.randint(self.size // 5, self.size // 2)
            cv2.rectangle(img, (wx, random.randint(0, self.size // 4)),
                          (min(self.size, wx + ww), y0 - random.randint(15, 50)), (155, 190, 205), -1)
            cv2.line(img, (wx + ww // 2, 0), (wx + ww // 2, y0), (70, 70, 65), max(2, self.size // 100))
        for _ in range(random.randint(2, 6)):
            x = random.randint(0, self.size)
            y = random.randint(self.size // 5, y0)
            cv2.rectangle(img, (x, y), (min(self.size, x + random.randint(self.size // 12, self.size // 3)),
                                        min(y0, y + random.randint(self.size // 12, self.size // 2))),
                          tuple(np.random.randint(35, 130, 3).tolist()), -1)
        return self._noise_layer(img, random.uniform(2, 7))

    def _nature_scene(self) -> np.ndarray:
        h = self.size
        sky = np.linspace(np.random.randint(120, 215, 3), np.random.randint(180, 245, 3), h).astype(np.uint8)
        img = np.repeat(sky[:, None, :], h, axis=1)
        horizon = random.randint(int(.35 * h), int(.68 * h))
        img[horizon:] = np.array([55, 92, 52], dtype=np.uint8)
        for _ in range(random.randint(2, 5)):
            pts = np.array([(0, horizon), *[(random.randint(0, h), random.randint(horizon - h // 8, horizon + h // 8))
                                             for _ in range(5)], (h, horizon)], np.int32)
            cv2.fillPoly(img, [pts], tuple(np.random.randint(45, 130, 3).tolist()))
        for _ in range(random.randint(5, 16)):
            x = random.randint(-h // 10, h)
            base = random.randint(int(.55 * h), h)
            cv2.rectangle(img, (x, base - random.randint(h // 5, h // 2)), (x + random.randint(2, h // 35), base),
                          (35, 55, 32), -1)
            for _ in range(random.randint(3, 9)):
                cx = x + random.randint(-h // 12, h // 12)
                cy = base - random.randint(h // 5, h // 2)
                cv2.ellipse(img, (cx, cy), (random.randint(h // 20, h // 8), random.randint(h // 30, h // 12)),
                            random.randint(0, 180), 0, 360, (30, 75, 35), -1)
        return self._noise_layer(img, random.uniform(3, 10))

    def _high_low_contrast(self, img: np.ndarray, strength: float = 1.0) -> np.ndarray:
        pil = Image.fromarray(img)
        if strength <= 0:
            return np.array(pil)
        lo, hi = 1.0 - 0.62 * strength, 1.0 + 1.0 * strength
        factor = random.uniform(1.0, hi) if random.random() < 0.5 else random.uniform(lo, 1.0)
        pil = ImageEnhance.Contrast(pil).enhance(factor)
        pil = ImageEnhance.Brightness(pil).enhance(random.uniform(1.0 - 0.28 * strength, 1.0 + 0.28 * strength))
        if random.random() < 0.25 * strength:
            pil = pil.filter(ImageFilter.GaussianBlur(random.uniform(0.2, 1.2 * strength)))
        return np.array(pil)

    def _real_photo(self) -> Optional[np.ndarray]:
        '''يحمّل صورة حقيقية عشوائية من BACKGROUND_DIR ويقتطع منها مربعاً بحجم self.size.
        يُرجع None بصمت عند أي خطأ قراءة (ملف تالف مثلاً) بدل إيقاف كامل التدريب.'''
        if not self.bg_paths:
            return None
        try:
            path = random.choice(self.bg_paths)
            pil = Image.open(path).convert("RGB")
            w, h = pil.size
            if w < self.size or h < self.size:
                scale = self.size / min(w, h)
                pil = pil.resize((max(self.size, int(w * scale)), max(self.size, int(h * scale))))
                w, h = pil.size
            x = random.randint(0, w - self.size)
            y = random.randint(0, h - self.size)
            crop = pil.crop((x, y, x + self.size, y + self.size))
            return np.array(crop)
        except Exception:
            return None

    def _aged_document(self) -> np.ndarray:
        '''وثيقة/نقش عتيق اصطناعي: ورق مصفرّ غير منتظم + بقع رطوبة + تفاوت إضاءة + خدوش دقيقة —
        يقرّب الفجوة بين الخلفيات الاصطناعية والصور الحقيقية للنصوص الأثرية (نقوش حجرية، مخطوطات قديمة).'''
        base = np.random.randint(150, 205, 3)
        img = np.full((self.size, self.size, 3), base, dtype=np.float32)

        grad = self._organic_texture(img.shape, scale=random.randint(3, 6), blur=self.size / 6)
        img += (grad[..., None] - 0.5) * random.uniform(40, 90)

        for _ in range(random.randint(6, 22)):
            x, y = random.randrange(self.size), random.randrange(self.size)
            r = random.randint(max(2, self.size // 30), max(3, self.size // 6))
            layer = np.zeros((self.size, self.size), np.uint8)
            cv2.circle(layer, (x, y), r, random.randint(30, 110), -1)
            layer = cv2.GaussianBlur(layer, (0, 0), max(1.0, r / random.uniform(2, 4))) / 255.0
            img -= layer[..., None] * random.uniform(15, 45)

        for _ in range(random.randint(3, 12)):
            x1, y1 = random.randrange(self.size), random.randrange(self.size)
            length = random.randint(max(2, self.size // 12), max(3, self.size // 3))
            angle = random.uniform(0, math.pi)
            x2 = int(x1 + length * math.cos(angle))
            y2 = int(y1 + length * math.sin(angle))
            shade = float(random.randint(90, 160))
            cv2.line(img, (x1, y1), (x2, y2), (shade, shade, shade), 1)

        img = np.clip(img, 0, 255).astype(np.uint8)
        return self._noise_layer(img, random.uniform(3, 9))

    def generate(self, complexity: float = 1.0) -> Image.Image:
        '''complexity=0 -> خلفية شبه مصمتة بسيطة جداً (بداية التعلّم التدريجي)
        complexity=1 -> كامل التنوع والتعقيد (نهاية التعلّم التدريجي / الاختبار).
        عند توفر خلفيات حقيقية في BACKGROUND_DIR تُخلط بنسبة متزايدة مع الصعوبة.'''
        complexity = float(np.clip(complexity, 0.0, 1.0))

        if self.bg_paths and random.random() < 0.15 + 0.35 * complexity:
            real = self._real_photo()
            if real is not None:
                real = self._high_low_contrast(real, strength=complexity)
                return Image.fromarray(real).convert("RGB")

        if complexity < 0.15:
            img = self._flat_paper()
        elif complexity < 0.4:
            img = random.choice([self._flat_paper, self._paper_texture])()
        elif complexity < 0.7:
            img = random.choice([self._paper_texture, self._indoor_gradient, self._aged_document])()
        else:
            img = random.choice([self._paper_texture, self._street_sign, self._indoor_gradient,
                                  self._nature_scene, self._aged_document])()
        img = self._high_low_contrast(img, strength=complexity)
        return Image.fromarray(img).convert("RGB")


print("تم تعريف: BackgroundGenerator (إجرائي + خلفية وثيقة عتيقة + مزج اختياري بخلفيات حقيقية)")


تم تعريف: BackgroundGenerator (إجرائي + خلفية وثيقة عتيقة + مزج اختياري بخلفيات حقيقية)


In [ ]:
@dataclass
class CharBox:
    text: str
    polygon: Polygon


@dataclass
class RenderedText:
    patch: Image.Image
    paste_xy: Tuple[int, int]
    char_boxes: List[CharBox] = field(default_factory=list)
    word_spans: List[Tuple[int, int]] = field(default_factory=list)


def order_quad_clockwise(poly: Polygon) -> Polygon:
    pts = np.array(poly, dtype=np.float64)
    cx, cy = pts.mean(axis=0)
    angles = np.arctan2(pts[:, 1] - cy, pts[:, 0] - cx)
    ordered = pts[np.argsort(angles)]
    start = int(np.argmin(ordered[:, 0] + ordered[:, 1]))
    ordered = np.roll(ordered, -start, axis=0)
    return [(float(x), float(y)) for x, y in ordered]


def _tight_ink_polygon(font: ImageFont.FreeTypeFont, cluster: str,
                        pen_x: float, baseline_y: float) -> Tuple[Polygon, float]:
    pad = max(font.size, 8)
    canvas_size = font.size * 3 + pad * 2
    origin = pad + font.size

    mask_img = Image.new("L", (canvas_size, canvas_size), 0)
    mask_draw = ImageDraw.Draw(mask_img)
    try:
        mask_draw.text((origin, origin), cluster, font=font, fill=255)
    except Exception:
        pass

    advance = font.getlength(cluster)
    if advance <= 0:
        advance = font.size * 0.5

    mask = np.array(mask_img)
    ys, xs = np.nonzero(mask > 10)

    if xs.size == 0:
        left, top, right, bottom = 0.0, -font.size * 0.6, max(advance, font.size * 0.3), font.size * 0.1
        polygon = [(pen_x + left, baseline_y + top), (pen_x + right, baseline_y + top),
                   (pen_x + right, baseline_y + bottom), (pen_x + left, baseline_y + bottom)]
        return polygon, advance

    pts = np.column_stack([xs, ys]).astype(np.float32)
    (cx, cy), (rw, rh), angle = cv2.minAreaRect(pts)
    box = cv2.boxPoints(((cx, cy), (rw, rh), angle))

    box_padded = []
    for (x, y) in box:
        dx, dy = x - cx, y - cy
        norm = math.hypot(dx, dy) or 1.0
        box_padded.append((x + dx / norm, y + dy / norm))

    polygon = [(pen_x + (x - origin), baseline_y + (y - origin)) for (x, y) in box_padded]
    return order_quad_clockwise(polygon), advance


def render_text_block(text: str, font: ImageFont.FreeTypeFont, fill=(0, 0, 0, 255)) -> RenderedText:
    lines = text.split("\n")
    line_height = int(font.size * 1.5)
    max_line_w = 0
    line_cluster_lists = []
    for line in lines:
        words = re.split(r"(\s+)", line)
        clusters_this_line = []
        for w in words:
            if w.strip() == "":
                clusters_this_line.append((None, w))
            else:
                clusters_this_line.append((split_graphemes(w), None))
        line_cluster_lists.append(clusters_this_line)

    tmp_draw = ImageDraw.Draw(Image.new("RGBA", (10, 10)))
    for line in lines:
        w = tmp_draw.textlength(line if line else " ", font=font)
        max_line_w = max(max_line_w, w)

    pad = font.size
    canvas_w = int(max_line_w) + pad * 2
    canvas_h = line_height * len(lines) + pad * 2
    layer = Image.new("RGBA", (canvas_w, canvas_h), (0, 0, 0, 0))
    draw = ImageDraw.Draw(layer)

    char_boxes: List[CharBox] = []
    word_spans: List[Tuple[int, int]] = []

    for li, clusters_this_line in enumerate(line_cluster_lists):
        pen_x = float(pad)
        baseline_y = float(pad + li * line_height)
        for clusters, space in clusters_this_line:
            if clusters is None:
                pen_x += tmp_draw.textlength(space, font=font)
                continue
            word_start = len(char_boxes)
            for cluster in clusters:
                draw.text((pen_x, baseline_y), cluster, font=font, fill=fill)
                polygon, advance = _tight_ink_polygon(font, cluster, pen_x, baseline_y)
                char_boxes.append(CharBox(text=cluster, polygon=polygon))
                pen_x += advance
            word_end = len(char_boxes) - 1
            if word_end >= word_start:
                word_spans.append((word_start, word_end))

    return RenderedText(patch=layer, paste_xy=(0, 0), char_boxes=char_boxes, word_spans=word_spans)


def rotate_rendered_text(rt: RenderedText, angle_deg: float) -> RenderedText:
    if abs(angle_deg) < 0.01:
        return rt
    w, h = rt.patch.size
    rotated = rt.patch.rotate(angle_deg, expand=True, resample=Image.BICUBIC)
    new_w, new_h = rotated.size
    cx, cy = w / 2.0, h / 2.0
    ncx, ncy = new_w / 2.0, new_h / 2.0
    theta = math.radians(-angle_deg)
    cos_t, sin_t = math.cos(theta), math.sin(theta)

    def transform(p: Point) -> Point:
        x, y = p[0] - cx, p[1] - cy
        xr = x * cos_t - y * sin_t
        yr = x * sin_t + y * cos_t
        return (xr + ncx, yr + ncy)

    new_boxes = []
    for cb in rt.char_boxes:
        new_poly = [transform(pt) for pt in cb.polygon]
        new_boxes.append(CharBox(text=cb.text, polygon=new_poly))

    return RenderedText(patch=rotated, paste_xy=rt.paste_xy, char_boxes=new_boxes, word_spans=rt.word_spans)


print("تم تعريف: render_text_block, rotate_rendered_text")


تم تعريف: render_text_block, rotate_rendered_text


In [ ]:
def apply_motion_blur(img: np.ndarray, k: int) -> np.ndarray:
    if k <= 1:
        return img
    kernel = np.zeros((k, k))
    angle = random.uniform(0, 180)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2, k / 2), angle, 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= kernel.sum() + 1e-8
    return cv2.filter2D(img, -1, kernel)


def apply_gaussian_noise(img: np.ndarray, sigma: float) -> np.ndarray:
    noise = np.random.normal(0, sigma, img.shape)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def augment_final_image(img: Image.Image, strength: float = 1.0) -> Image.Image:
    '''strength في [0, 1] يتحكّم بشدّة كل التمويهات/الضوضاء — يُستخدم في
    التعلّم التدريجي (بداية سهلة بلا تشويش تقريباً -> نهاية بتشويش كامل).'''
    strength = float(np.clip(strength, 0.0, 1.0))
    arr = np.array(img.convert("RGB"))
    if random.random() < 0.5 * strength:
        arr = apply_gaussian_noise(arr, sigma=random.uniform(2, 10) * strength)
    if random.random() < 0.35 * strength:
        arr = apply_motion_blur(arr, k=random.choice([3, 5, 7]))
    pil = Image.fromarray(arr)
    if random.random() < 0.5 * strength:
        pil = ImageEnhance.Contrast(pil).enhance(random.uniform(1 - 0.2 * strength, 1 + 0.3 * strength))
    if random.random() < 0.5 * strength:
        pil = ImageEnhance.Brightness(pil).enhance(random.uniform(1 - 0.2 * strength, 1 + 0.2 * strength))
    if random.random() < 0.2 * strength:
        pil = pil.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 0.8) * strength))
    return pil


def is_overlapping(box_a, box_b, margin: int = 12) -> bool:
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ax1, ay1, ax2, ay2 = ax1 - margin, ay1 - margin, ax2 + margin, ay2 + margin
    return not (ax2 < bx1 or ax1 > bx2 or ay2 < by1 or ay1 > by2)


def clamp_polygon_to_image(poly: Polygon, w: int, h: int) -> Polygon:
    return [(min(max(x, 0), w - 1), min(max(y, 0), h - 1)) for x, y in poly]


print("تم تعريف: augment_final_image, is_overlapping, clamp_polygon_to_image")


تم تعريف: augment_final_image, is_overlapping, clamp_polygon_to_image


In [ ]:
def generate_one_sample_memory(idx, img_size, font_mgr, bg_gen, corpus, difficulty):
    '''يولّد عينة واحدة بالكامل في الذاكرة (بلا أي كتابة على القرص) وفق
    قاموس difficulty القادم من دالة التعلّم التدريجي curriculum_stage().'''
    bg = bg_gen.generate(complexity=difficulty["bg_complexity"]).resize((img_size, img_size)).convert("RGBA")
    chars, pairs, placed_boxes = [], [], []

    n_instances = random.randint(*difficulty["n_instances"])
    for _ in range(n_instances):
        text, _kind = corpus.sample(weights=difficulty["corpus_weights"])
        if random.random() < difficulty.get("archaic_prob", 0.0):
            text = corpus.inject_archaic(text)
        f_lo, f_hi = difficulty["font_size_frac"]
        font_size = random.randint(max(10, int(img_size * f_lo)), max(11, int(img_size * f_hi)))
        font = font_mgr.random_font(font_size)
        fill = random.choice([(0, 0, 0, 255), (20, 20, 20, 255), (255, 255, 255, 255), (230, 230, 230, 255)])

        angle = random.uniform(-difficulty["rotation_max"], difficulty["rotation_max"])
        rt = rotate_rendered_text(render_text_block(text, font, fill=fill), angle)

        pw, ph = rt.patch.size
        if pw >= img_size or ph >= img_size:
            continue

        placed = False
        for _attempt in range(40):
            x = random.randint(0, img_size - pw)
            y = random.randint(0, img_size - ph)
            box = (x, y, x + pw, y + ph)
            if not any(is_overlapping(box, old) for old in placed_boxes):
                placed_boxes.append(box)
                paste_x, paste_y = x, y
                placed = True
                break
        if not placed:
            continue

        bg.alpha_composite(rt.patch, dest=(paste_x, paste_y))
        base_id = len(chars)
        for cb in rt.char_boxes:
            poly = clamp_polygon_to_image([(x + paste_x, y + paste_y) for x, y in cb.polygon], img_size, img_size)
            chars.append({"character_id": len(chars), "text": cb.text,
                          "polygon": [[round(x, 2), round(y, 2)] for x, y in poly]})
        for s, e in rt.word_spans:
            for a, b in zip(range(s, e), range(s + 1, e + 1)):
                pairs.append({"from": base_id + a, "to": base_id + b})

    final_img = augment_final_image(bg.convert("RGB"), strength=difficulty["aug_strength"])
    return final_img, {"image_id": int(idx), "width": img_size, "height": img_size,
                        "characters": chars, "affinity_pairs": pairs}


print("تم تعريف: generate_one_sample_memory (تدريجي بالكامل + حَقن الحروف الأثرية)")


تم تعريف: generate_one_sample_memory (تدريجي بالكامل + حَقن الحروف الأثرية)


## 3) نموذج CRAFT (VGG16-BN + U-Net Decoder)

نفس معمارية CRAFT الرسمية تماماً (لضمان توافق مفاتيح الأوزان مع الملف المُدرَّب مسبقاً `craft_mlt_25k.pth`).
الأوزان الفعلية التي يبدأ منها التدريب تُحدَّد لاحقاً في خلية "الإعدادات الفائقة" بالترتيب التالي:
1) checkpoint محفوظ من جلسة سابقة لهذا الدفتر (استئناف)، وإلا 2) أوزان CRAFT الرسمية المُدرَّبة مسبقاً (تخصيص/Fine-tuning)،
وإلا 3) أوزان عشوائية بالكامل (نفس سلوك الإصدار السابق، fallback أخير فقط).


In [ ]:
from collections import namedtuple

import torch
import torch.nn as nn
import torch.nn.init as init
from torchvision import models


def init_weights(modules):
    for m in modules:
        if isinstance(m, nn.Conv2d):
            init.xavier_uniform_(m.weight.data)
            if m.bias is not None:
                m.bias.data.zero_()
        elif isinstance(m, nn.BatchNorm2d):
            m.weight.data.fill_(1)
            m.bias.data.zero_()
        elif isinstance(m, nn.Linear):
            m.weight.data.normal_(0, 0.01)
            m.bias.data.zero_()


class vgg16_bn(nn.Module):
    def __init__(self, pretrained=False, freeze=False):
        super().__init__()
        vgg_pretrained_features = models.vgg16_bn(weights=None).features
        self.slice1 = nn.Sequential()
        self.slice2 = nn.Sequential()
        self.slice3 = nn.Sequential()
        self.slice4 = nn.Sequential()
        self.slice5 = nn.Sequential()
        for x in range(12):
            self.slice1.add_module(str(x), vgg_pretrained_features[x])
        for x in range(12, 19):
            self.slice2.add_module(str(x), vgg_pretrained_features[x])
        for x in range(19, 29):
            self.slice3.add_module(str(x), vgg_pretrained_features[x])
        for x in range(29, 39):
            self.slice4.add_module(str(x), vgg_pretrained_features[x])

        self.slice5 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(512, 1024, kernel_size=3, padding=6, dilation=6),
            nn.Conv2d(1024, 1024, kernel_size=1),
        )

        init_weights(self.slice1.modules())
        init_weights(self.slice2.modules())
        init_weights(self.slice3.modules())
        init_weights(self.slice4.modules())
        init_weights(self.slice5.modules())

        if freeze:
            for param in self.slice1.parameters():
                param.requires_grad = False

    def forward(self, X):
        h = self.slice1(X)
        h_relu2_2 = h
        h = self.slice2(h)
        h_relu3_2 = h
        h = self.slice3(h)
        h_relu4_3 = h
        h = self.slice4(h)
        h_relu5_3 = h
        h = self.slice5(h)
        h_fc7 = h
        vgg_outputs = namedtuple("VggOutputs", ["fc7", "relu5_3", "relu4_3", "relu3_2", "relu2_2"])
        return vgg_outputs(h_fc7, h_relu5_3, h_relu4_3, h_relu3_2, h_relu2_2)


print("تم تعريف: vgg16_bn (pretrained=False دائماً — تدريب من الصفر)")


تم تعريف: vgg16_bn (pretrained=False دائماً — تدريب من الصفر)


In [ ]:
import torch.nn.functional as F


class double_conv(nn.Module):
    def __init__(self, in_ch, mid_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + mid_ch, mid_ch, kernel_size=1),
            nn.BatchNorm2d(mid_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class CRAFT(nn.Module):
    def __init__(self, pretrained=False, freeze=False):
        super().__init__()
        self.basenet = vgg16_bn(pretrained=pretrained, freeze=freeze)

        self.upconv1 = double_conv(1024, 512, 256)
        self.upconv2 = double_conv(512, 256, 128)
        self.upconv3 = double_conv(256, 128, 64)
        self.upconv4 = double_conv(128, 64, 32)

        self.conv_cls = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 16, kernel_size=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 2, kernel_size=1),
        )

        init_weights(self.upconv1.modules())
        init_weights(self.upconv2.modules())
        init_weights(self.upconv3.modules())
        init_weights(self.upconv4.modules())
        init_weights(self.conv_cls.modules())

    def forward(self, x):
        sources = self.basenet(x)

        y = torch.cat([sources[0], sources[1]], dim=1)
        y = self.upconv1(y)

        y = F.interpolate(y, size=sources[2].size()[2:], mode="bilinear", align_corners=False)
        y = torch.cat([y, sources[2]], dim=1)
        y = self.upconv2(y)

        y = F.interpolate(y, size=sources[3].size()[2:], mode="bilinear", align_corners=False)
        y = torch.cat([y, sources[3]], dim=1)
        y = self.upconv3(y)

        y = F.interpolate(y, size=sources[4].size()[2:], mode="bilinear", align_corners=False)
        y = torch.cat([y, sources[4]], dim=1)
        feature = self.upconv4(y)

        y = self.conv_cls(feature)
        # المخرجات logits خام بشكل [B, H, W, 2] — القناة 0: Region, القناة 1: Affinity
        return y.permute(0, 2, 3, 1), feature


print("تم تعريف: CRAFT")


تم تعريف: CRAFT


## 4) دالة الخسارة CraftLoss (مع OHEM + sigmoid داخلي)

ملاحظة إصلاح مهمة: مخرجات النموذج logits خام (بلا sigmoid)، بينما الأهداف (خرائط Gaussian) في المجال [0, 1].
لذلك يجب تطبيق `sigmoid` داخل الخسارة قبل المقارنة — وهذا ما يفعله الإصدار أدناه (وهو الإصدار الوحيد والموحّد المستخدم في كل الدفتر لتفادي أي تعارض بين نسخ متعددة).


In [ ]:
class CraftLoss(nn.Module):
    def __init__(self, ohem_ratio=3, use_ohem=True):
        super().__init__()
        self.mse = nn.MSELoss(reduction="none")
        self.ohem_ratio = ohem_ratio
        self.use_ohem = use_ohem

    def _ohem_single(self, pred, target):
        positive_mask = target > 0.1
        num_positive = int(positive_mask.sum().item())
        pixel_loss = self.mse(pred, target)

        if num_positive == 0:
            num_negative = min(100, pixel_loss.numel())
            top_negative, _ = torch.topk(pixel_loss.reshape(-1), num_negative)
            return top_negative.mean()

        positive_loss = pixel_loss[positive_mask]
        negative_mask = ~positive_mask
        num_negative = min(int(num_positive * self.ohem_ratio), int(negative_mask.sum().item()))
        if num_negative > 0:
            top_negative, _ = torch.topk(pixel_loss[negative_mask], num_negative)
            return torch.cat([positive_loss, top_negative]).mean()
        return positive_loss.mean()

    def forward(self, pred, target):
        '''pred: [B,H,W,2] logits من CRAFT.forward، target: [B,2,H,W] في [0,1].'''
        if pred.ndim != 4 or target.ndim != 4:
            raise ValueError(f"pred وtarget يجب أن يكونا 4D: pred={pred.shape}, target={target.shape}")

        if pred.shape[-1] == 2:
            pred = pred.permute(0, 3, 1, 2).contiguous()
        elif pred.shape[1] != 2:
            raise ValueError(f"شكل pred غير مدعوم: {pred.shape}")

        if target.shape[-1] == 2 and target.shape[1] != 2:
            target = target.permute(0, 3, 1, 2).contiguous()
        elif target.shape[1] != 2:
            raise ValueError(f"شكل target غير مدعوم: {target.shape}")

        if pred.shape != target.shape:
            raise RuntimeError(f"عدم تطابق pred وtarget: pred={pred.shape}, target={target.shape}")

        pred = torch.sigmoid(pred)
        pred_region, pred_affinity = pred[:, 0], pred[:, 1]
        target_region, target_affinity = target[:, 0], target[:, 1]

        if not self.use_ohem:
            return self.mse(pred_region, target_region).mean() + self.mse(pred_affinity, target_affinity).mean()

        total = pred.new_tensor(0.0)
        for i in range(pred.shape[0]):
            total = total + self._ohem_single(pred_region[i], target_region[i])
            total = total + self._ohem_single(pred_affinity[i], target_affinity[i])
        return total / pred.shape[0]


print("تم تعريف: CraftLoss (نسخة موحّدة وحيدة في كل الدفتر)")


تم تعريف: CraftLoss (نسخة موحّدة وحيدة في كل الدفتر)


## 5) تحويل العينات إلى Tensor + خرائط Gaussian (Region/Affinity)


In [ ]:
from skimage import io as _skio


def normalizeMeanVariance(in_img, mean=(0.485, 0.456, 0.406), variance=(0.229, 0.224, 0.225)):
    img = in_img.copy().astype(np.float32)
    img -= np.array([mean[0] * 255.0, mean[1] * 255.0, mean[2] * 255.0], dtype=np.float32)
    img /= np.array([variance[0] * 255.0, variance[1] * 255.0, variance[2] * 255.0], dtype=np.float32)
    return img


def denormalizeMeanVariance(in_img, mean=(0.485, 0.456, 0.406), variance=(0.229, 0.224, 0.225)):
    img = in_img.copy()
    img *= variance
    img += mean
    img *= 255.0
    return np.clip(img, 0, 255).astype(np.uint8)


def resize_aspect_ratio(img, square_size, interpolation, mag_ratio=1):
    height, width, channel = img.shape
    target_size = mag_ratio * max(height, width)
    if target_size > square_size:
        target_size = square_size
    ratio = target_size / max(height, width)
    target_h, target_w = int(height * ratio), int(width * ratio)
    proc = cv2.resize(img, (target_w, target_h), interpolation=interpolation)

    target_h32, target_w32 = target_h, target_w
    if target_h % 32 != 0:
        target_h32 = target_h + (32 - target_h % 32)
    if target_w % 32 != 0:
        target_w32 = target_w + (32 - target_w % 32)
    resized = np.zeros((target_h32, target_w32, channel), dtype=np.float32)
    resized[0:target_h, 0:target_w, :] = proc
    size_heatmap = (int(target_w32 / 2), int(target_h32 / 2))
    return resized, ratio, size_heatmap


def cvt2HeatmapImg(img):
    img = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    return cv2.applyColorMap(img, cv2.COLORMAP_JET)


class GaussianTransformer:
    def __init__(self, img_size=200, sigma=40):
        self.img_size = img_size
        self.sigma = sigma
        self.kernel = self._generate_kernel()

    def _generate_kernel(self):
        x = np.arange(0, self.img_size, 1, float)
        y = x[:, np.newaxis]
        x0 = y0 = self.img_size // 2
        kernel = np.exp(-((x - x0) ** 2 + (y - y0) ** 2) / (2 * self.sigma ** 2))
        return kernel / np.max(kernel)

    def render(self, target_size, polygons):
        target = np.zeros(target_size, dtype=np.float32)
        if not polygons:
            return target
        src_pts = np.array([[0, 0], [self.img_size, 0], [self.img_size, self.img_size], [0, self.img_size]],
                            dtype=np.float32)
        for poly in polygons:
            poly = np.array(poly).astype(np.float32)
            if len(poly) != 4:
                continue
            M = cv2.getPerspectiveTransform(src_pts, poly)
            warped = cv2.warpPerspective(self.kernel, M, (target_size[1], target_size[0]))
            target = np.maximum(target, warped)
        return target


def get_affinity_poly(p1, p2):
    return np.array([
        (p1[0] + p1[1]) / 2.0,
        (p2[0] + p2[1]) / 2.0,
        (p2[2] + p2[3]) / 2.0,
        (p1[2] + p1[3]) / 2.0,
    ], dtype=np.float32)


_GT_ENGINE = GaussianTransformer()


def sample_to_tensor(image, ann, target_size=768):
    img = np.asarray(image.convert("RGB"))
    img_res, ratio, _ = resize_aspect_ratio(img, target_size, interpolation=cv2.INTER_LINEAR)
    image_tensor = torch.from_numpy(normalizeMeanVariance(img_res)).permute(2, 0, 1).float()
    h, w = img_res.shape[:2]
    map_size = (h // 2, w // 2)
    sx = sy = ratio * 0.5

    char_polys, affinity_polys, chars_dict = [], [], {}
    for char in ann.get("characters", []):
        p = np.asarray(char["polygon"], dtype=np.float32).copy()
        p[:, 0] *= sx
        p[:, 1] *= sy
        char_polys.append(p)
        chars_dict[char["character_id"]] = p
    for pair in ann.get("affinity_pairs", []):
        p1, p2 = chars_dict.get(pair["from"]), chars_dict.get(pair["to"])
        if p1 is not None and p2 is not None:
            affinity_polys.append(get_affinity_poly(p1, p2))

    target = np.zeros((2, map_size[0], map_size[1]), dtype=np.float32)
    target[0] = _GT_ENGINE.render(map_size, char_polys)
    target[1] = _GT_ENGINE.render(map_size, affinity_polys)
    return image_tensor, torch.from_numpy(target)


def generate_online_batch(batch_size, step, img_size, target_size, font_mgr, bg_gen, corpus, difficulty):
    xs, ys = [], []
    for j in range(batch_size):
        image, ann = generate_one_sample_memory(step * batch_size + j, img_size, font_mgr, bg_gen, corpus, difficulty)
        x, y = sample_to_tensor(image, ann, target_size)
        xs.append(x)
        ys.append(y)
    return torch.stack(xs), torch.stack(ys)


print("تم تعريف: sample_to_tensor, generate_online_batch")


تم تعريف: sample_to_tensor, generate_online_batch


## 6) التعلّم التدريجي (Curriculum Learning) — من السهل جداً إلى الصعب الكامل

بما أن التدريب يفترض دائماً أن النموذج **يبدأ من الصفر**، فإن أفضل استراتيجية هي البدء بعيّنات
سهلة جداً على النموذج (خلفية شبه مصمتة، كتلة نص واحدة كبيرة، بلا دوران، بلا ضوضاء) حتى تتشكّل
إشارة تعلّم واضحة (loss ينخفض بثبات)، ثم زيادة الصعوبة تدريجياً عبر 4 مراحل حتى الوصول للواقعية الكاملة.

**التحقق (Validation) دائماً يستخدم أصعب مستوى صعوبة** (المرحلة الأخيرة) بغض النظر عن مرحلة التدريب
الحالية، لضمان مقياس مقارنة عادل وثابت طوال التدريب (بدل أن يتحسّن الرقم فقط لأن البيانات سهلت).


In [ ]:
def curriculum_stage(step: int, total_steps: int) -> dict:
    """يرجع قاموس معاملات الصعوبة بحسب نسبة التقدّم في التدريب.
    CURRICULUM_PROGRESS_OFFSET (يُعرَّف في خلية الإعدادات الفائقة) يزيح نقطة البداية إلى الأمام:
    بما أن التدريب الآن تخصيص (Fine-tuning) فوق أوزان مُدرَّبة مسبقاً وليس بدءاً من الصفر، لا حاجة
    لقضاء وقت في المرحلة الأسهل جداً — القيمة الافتراضية 0.0 تحافظ على السلوك الأصلي إن رغبت بذلك."""
    raw = 0.0 if total_steps <= 0 else min(1.0, step / total_steps)
    offset = float(globals().get("CURRICULUM_PROGRESS_OFFSET", 0.0))
    progress = offset + (1.0 - offset) * raw

    if progress < 0.15:  # المرحلة 1: بداية شديدة السهولة
        return {
            "name": "1-بداية سهلة جداً",
            "n_instances": (1, 2),
            "font_size_frac": (0.11, 0.16),
            "rotation_max": 2.0,
            "bg_complexity": 0.08,
            "aug_strength": 0.05,
            "corpus_weights": {"word": 75, "sentence": 5, "digits": 15, "multiline": 5},
            "archaic_prob": 0.08,
        }
    if progress < 0.40:  # المرحلة 2
        return {
            "name": "2-صعوبة متوسطة-منخفضة",
            "n_instances": (1, 3),
            "font_size_frac": (0.07, 0.13),
            "rotation_max": 7.0,
            "bg_complexity": 0.35,
            "aug_strength": 0.35,
            "corpus_weights": {"word": 50, "sentence": 20, "digits": 15, "multiline": 15},
            "archaic_prob": 0.14,
        }
    if progress < 0.70:  # المرحلة 3
        return {
            "name": "3-صعوبة متوسطة-عالية",
            "n_instances": (2, 4),
            "font_size_frac": (0.05, 0.11),
            "rotation_max": 12.0,
            "bg_complexity": 0.7,
            "aug_strength": 0.7,
            "corpus_weights": {"word": 40, "sentence": 28, "digits": 15, "multiline": 17},
            "archaic_prob": 0.20,
        }
    # المرحلة 4: الواقعية الكاملة (وهي أيضاً إعداد التحقق/الاختبار الثابت)
    return {
        "name": "4-واقعية كاملة",
        "n_instances": (2, 6),
        "font_size_frac": (0.03, 0.10),
        "rotation_max": 15.0,
        "bg_complexity": 1.0,
        "aug_strength": 1.0,
        "corpus_weights": {"word": 35, "sentence": 30, "digits": 15, "multiline": 20},
        "archaic_prob": 0.28,
    }


def hardest_difficulty() -> dict:
    """مستوى الصعوبة الثابت المستخدم دائماً في Validation (المرحلة 4) —
    يضمن مقارنة عادلة لقيمة الخسارة عبر كل خطوات التدريب."""
    return curriculum_stage(total_steps=1, step=1)


print("تم تعريف: curriculum_stage, hardest_difficulty (+ archaic_prob في كل مرحلة)")
for _p in [0.0, 0.15, 0.4, 0.7, 0.99]:
    _d = curriculum_stage(int(_p * 1000), 1000)
    print(f"  تقدّم خام={_p:>4} -> {_d['name']} | archaic_prob={_d['archaic_prob']}")


تم تعريف: curriculum_stage, hardest_difficulty (+ archaic_prob في كل مرحلة)
  تقدّم خام= 0.0 -> 1-بداية سهلة جداً | archaic_prob=0.08
  تقدّم خام=0.15 -> 2-صعوبة متوسطة-منخفضة | archaic_prob=0.14
  تقدّم خام= 0.4 -> 3-صعوبة متوسطة-عالية | archaic_prob=0.2
  تقدّم خام= 0.7 -> 4-واقعية كاملة | archaic_prob=0.28
  تقدّم خام=0.99 -> 4-واقعية كاملة | archaic_prob=0.28


## 7) الإعدادات الفائقة والجهاز


In [ ]:
import torch

SEED = 20260901

BATCH_SIZE = 4
IMG_SIZE = 800
TARGET_SIZE = 768

# جديد: استخدام أوزان CRAFT الرسمية المُدرَّبة مسبقاً كنقطة انطلاق (تخصيص) بدلاً من العشوائية الكاملة.
# اضبطها على False فقط إن رغبت فعلياً بالعودة إلى سلوك "تدريب من الصفر" القديم.
USE_PRETRAINED_INIT = True

# إجمالي خطوات "منحنى التعلّم التدريجي" الكامل. كان 20000 عند التدريب من الصفر؛ التخصيص فوق أوزان
# مُدرَّبة مسبقاً يحتاج عادة أقل بكثير لأن الشبكة تعرف "الحرف" عموماً بالفعل وتحتاج فقط تكييفاً
# على شكل الحروف التايلاندية. غيّرها بحرية حسب ما تلاحظه من منحنى val_loss.
TOTAL_CURRICULUM_STEPS = 10000
STEPS_THIS_SESSION = 5000

# جديد: يزيح بداية التعلّم التدريجي إلى مستوى صعوبة أعلى مباشرة (0.0 = نفس السلوك القديم من الصفر،
# 0.3 = تخطّي المرحلة الأسهل تماماً لأن الشبكة الأساسية مُدرَّبة مسبقاً بالفعل ولا تحتاجها).
CURRICULUM_PROGRESS_OFFSET = 0.3

# جديد: معدل تعلّم أقل من التدريب من الصفر (كان 1e-4) — مناسب للتخصيص فوق أوزان جيدة أصلاً كي لا
# "نُخرِّب" ما تعلّمته الشبكة عمومياً عن الحروف. إن رأيت val_loss يهتز/يتذبذب بشدة في البداية، انزل
# إلى 2e-5. إن رأيته يتحسّن ببطء شديد بعد آلاف الخطوات، يمكن رفعه تدريجياً نحو 1e-4.
LEARNING_RATE = 5e-5

SAVE_EVERY_STEPS = 20        # حفظ آخر نموذج
VALIDATE_EVERY_STEPS = 100   # اختبار كل فترة ثابتة (Validation)
PREVIEW_EVERY_STEPS = 100     # اختبار بصري كل فترة ثابتة
VALIDATION_BATCHES = 5
VALIDATION_SEED = SEED + 10_000_000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("الجهاز المستخدم:", device)
if torch.cuda.is_available():
    print("اسم GPU:", torch.cuda.get_device_name(0))
print(f"خطط التدريب: {STEPS_THIS_SESSION} خطوة في هذه الجلسة، من إجمالي منحنى تدريجي طوله "
      f"{TOTAL_CURRICULUM_STEPS} خطوة (بإزاحة بداية = {CURRICULUM_PROGRESS_OFFSET}).")
print(f"تهيئة الأوزان: {'CRAFT مُدرَّب مسبقاً (تخصيص)' if USE_PRETRAINED_INIT else 'عشوائية بالكامل (من الصفر)'} "
      "— ما لم يوجد checkpoint سابق لهذا الدفتر فيُستأنف منه بدلاً من ذلك.")


الجهاز المستخدم: cuda
اسم GPU: Tesla T4
خطط التدريب: 5000 خطوة في هذه الجلسة، من إجمالي منحنى تدريجي طوله 10000 خطوة (بإزاحة بداية = 0.3).
تهيئة الأوزان: CRAFT مُدرَّب مسبقاً (تخصيص) — ما لم يوجد checkpoint سابق لهذا الدفتر فيُستأنف منه بدلاً من ذلك.


In [ ]:
random.seed(SEED)
np.random.seed(SEED % (2 ** 32 - 1))
torch.manual_seed(SEED)

font_mgr = FontManager(str(FONT_DIR))
bg_gen = BackgroundGenerator(IMG_SIZE, background_dir=str(BACKGROUND_DIR))  # إجرائي + مزج اختياري بخلفيات حقيقية
corpus = CorpusGenerator(corpus_file=None)

print("تم تجهيز: الخطوط، القاموس، مولّد الخلفيات الإجرائي.")


[BackgroundGenerator] تم العثور على 685 صورة خلفية حقيقية — ستُخلط مع الخلفيات الإجرائية.
[CorpusGenerator] تم تحميل 60073 كلمة من pythainlp.corpus.thai_words()
تم تجهيز: الخطوط، القاموس، مولّد الخلفيات الإجرائي.


In [ ]:
def load_pretrained_craft_weights(model: nn.Module, weights_path: Path, device) -> dict:
    """يحمّل أوزان CRAFT الرسمية (craft_mlt_25k.pth) فوق النموذج الحالي. يتعامل مع بادئة
    'module.' (ناتجة عن تدريب أصلي متعدد GPU) ويتجاهل بصمت أي طبقة بشكل/اسم غير مطابق
    (strict=False) بدلاً من رمي استثناء يوقف الدفتر بالكامل — لأن بعض الفروقات الطفيفة
    محتملة نظرياً ولا ينبغي أن توقف التخصيص، لكنها تُطبَع بوضوح ليعرف المستخدم ما جرى."""
    raw = torch.load(weights_path, map_location=device, weights_only=False)
    if isinstance(raw, dict) and "model" in raw and isinstance(raw["model"], dict):
        raw = raw["model"]  # في حال كان الملف checkpoint كاملاً وليس state_dict مباشرة

    cleaned = {}
    for k, v in raw.items():
        name = k[7:] if k.startswith("module.") else k
        cleaned[name] = v

    model_state = model.state_dict()
    matched = {k: v for k, v in cleaned.items() if k in model_state and model_state[k].shape == v.shape}
    missing = sorted(set(model_state.keys()) - set(matched.keys()))
    unexpected = sorted(set(cleaned.keys()) - set(model_state.keys()))

    model_state.update(matched)
    model.load_state_dict(model_state)

    return {"matched": len(matched), "total": len(model_state), "missing": missing, "unexpected": unexpected}


model = CRAFT(pretrained=False, freeze=False).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = CraftLoss()
start_step = 0
best_val_loss = float("inf")
best_step = 0

def _valid_checkpoint(path: Path) -> bool:
    return path.exists() and path.stat().st_size > 0

if _valid_checkpoint(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    required = {"model", "optimizer"}
    missing = required - set(checkpoint)
    if missing:
        raise RuntimeError(f"الـ checkpoint ناقص المفاتيح: {missing}")
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    start_step = int(checkpoint.get("step", 0))
    best_val_loss = float(checkpoint.get("best_val_loss", float("inf")))
    best_step = int(checkpoint.get("best_step", 0))
    print(f"[استئناف] تم العثور على checkpoint سابق لهذا الدفتر. البداية من Step={start_step}، "
          f"أفضل val_loss محفوظ={best_val_loss}")
elif USE_PRETRAINED_INIT and _valid_checkpoint(PRETRAINED_CRAFT_PATH):
    report = load_pretrained_craft_weights(model, PRETRAINED_CRAFT_PATH, device)
    print(f"[تخصيص] تم تحميل {report['matched']}/{report['total']} مصفوفة أوزان من "
          f"CRAFT المُدرَّب مسبقاً ({PRETRAINED_CRAFT_PATH.name}).")
    if report["missing"]:
        print(f"  لم تُحمَّل {len(report['missing'])} مصفوفة (ستبقى بتهيئة عشوائية) — طبيعي إن كان العدد صغيراً "
              "(مثلاً بسبب فروق دقيقة في التسمية)، لكن راجع القائمة إن كان العدد كبيراً:")
        print("   ", report["missing"][:10], "..." if len(report["missing"]) > 10 else "")
    if report["unexpected"]:
        print(f"  {len(report['unexpected'])} مصفوفة في الملف لم تُستخدَم (طبيعي، مثل طبقات VGG المصنِّفة الأصلية).")
    print("  هذا تدريب تخصيص (Fine-tuning) حقيقي فوق هذه الأوزان — وليس تدريباً من الصفر.")
else:
    reason = "USE_PRETRAINED_INIT=False" if not USE_PRETRAINED_INIT else f"الملف غير موجود: {PRETRAINED_CRAFT_PATH}"
    print(f"[من الصفر] لا يوجد checkpoint سابق ولا أوزان مُدرَّبة مسبقاً صالحة ({reason}) "
          "— يبدأ النموذج بأوزان عشوائية تماماً. شغّل خلية تنزيل الأوزان المُدرَّبة مسبقاً أعلاه لتفادي هذا.")

print("عدد بارامترات النموذج:", sum(p.numel() for p in model.parameters()))


[استئناف] تم العثور على checkpoint سابق لهذا الدفتر. البداية من Step=980، أفضل val_loss محفوظ=0.3693422555923462
عدد بارامترات النموذج: 20770466


In [ ]:
# اختبار آمن قبل بدء التدريب: Batch واحد + forward فقط (يستخدم أسهل مرحلة تدريجية)
_sanity_difficulty = curriculum_stage(start_step, TOTAL_CURRICULUM_STEPS)
random.seed(SEED + start_step)
np.random.seed((SEED + start_step) % (2 ** 32 - 1))

_test_x, _test_y = generate_online_batch(BATCH_SIZE, start_step, IMG_SIZE, TARGET_SIZE,
                                          font_mgr, bg_gen, corpus, _sanity_difficulty)
print("مرحلة الصعوبة الحالية:", _sanity_difficulty["name"])
print("شكل الصور:", tuple(_test_x.shape), "| شكل الأهداف:", tuple(_test_y.shape))

with torch.no_grad():
    _test_out, _ = model(_test_x.to(device))
print("شكل مخرجات النموذج:", tuple(_test_out.shape))

del _test_x, _test_y, _test_out
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("اجتاز الاختبار الآمن بنجاح.")


مرحلة الصعوبة الحالية: 2-صعوبة متوسطة-منخفضة
شكل الصور: (4, 3, 768, 768) | شكل الأهداف: (4, 2, 384, 384)
شكل مخرجات النموذج: (4, 384, 384, 2)
اجتاز الاختبار الآمن بنجاح.


## 8) أدوات التحقق (Validation) والمعاينة البصرية


In [ ]:
import matplotlib.pyplot as plt


def verify_batch(images, targets):
    assert images.ndim == 4, f"شكل الصور غير صحيح: {images.shape}"
    assert targets.ndim == 4, f"شكل الأهداف غير صحيح: {targets.shape}"
    assert images.shape[0] == targets.shape[0], f"اختلاف حجم Batch: {images.shape} مقابل {targets.shape}"
    assert images.shape[1] == 3, f"يجب أن تحتوي الصور على 3 قنوات: {images.shape}"
    assert targets.shape[1] == 2, f"يجب أن تحتوي الأهداف على قناتين: {targets.shape}"
    assert torch.isfinite(images).all(), "الصور تحتوي على NaN أو Inf"
    assert torch.isfinite(targets).all(), "الأهداف تحتوي على NaN أو Inf"


def outputs_to_bhwc(outputs):
    if outputs.ndim != 4:
        raise ValueError(f"شكل outputs غير صحيح: {outputs.shape}")
    if outputs.shape[-1] == 2:
        return outputs
    if outputs.shape[1] == 2:
        return outputs.permute(0, 2, 3, 1).contiguous()
    raise ValueError(f"شكل outputs غير مدعوم: {outputs.shape}")


def extract_prediction_boxes(region_map, threshold=0.5, min_area=3):
    binary = (region_map >= threshold).astype(np.uint8) * 255
    number, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    height, width = binary.shape
    image_area = height * width
    boxes = []
    for label_id in range(1, number):
        x, y, box_width, box_height, area = stats[label_id]
        if area < min_area or area > image_area * 0.30:
            continue
        if box_width >= width * 0.95 or box_height >= height * 0.95:
            continue
        boxes.append({"x1": int(x), "y1": int(y), "x2": int(x + box_width), "y2": int(y + box_height),
                      "area": int(area)})
    boxes.sort(key=lambda b: (b["y1"], b["x1"]))
    return boxes


@torch.no_grad()
def run_validation(num_batches: int) -> float:
    '''يقيس الخسارة دائماً على أصعب مستوى صعوبة ثابت (hardest_difficulty)
    وبذور عشوائية ثابتة -> رقم قابل للمقارنة عبر كل خطوات التدريب.'''
    was_training = model.training
    model.eval()
    difficulty = hardest_difficulty()
    losses = []
    try:
        for i in range(num_batches):
            seed = VALIDATION_SEED + i
            random.seed(seed)
            np.random.seed(seed % (2 ** 32 - 1))
            torch.manual_seed(seed)

            val_images, val_targets = generate_online_batch(BATCH_SIZE, i, IMG_SIZE, TARGET_SIZE,
                                                              font_mgr, bg_gen, corpus, difficulty)
            verify_batch(val_images, val_targets)
            val_images = val_images.to(device, non_blocking=True)
            val_targets = val_targets.to(device, non_blocking=True)

            val_outputs, _ = model(val_images)
            val_loss = criterion(val_outputs, val_targets)
            if not torch.isfinite(val_loss):
                raise RuntimeError(f"Validation Loss غير صالح: {val_loss.item()}")
            losses.append(float(val_loss.detach().cpu()))

            del val_images, val_targets, val_outputs, val_loss
        return sum(losses) / max(1, len(losses))
    finally:
        if was_training:
            model.train()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


@torch.no_grad()
def save_prediction_preview(images, outputs, step, sample_index=0, threshold=0.5, min_area=3, tag="train"):
    '''يرسم 3 لوحات: الصورة الأصلية، صناديق توقع النموذج، خريطة Region.
    يعرض داخل الدفتر ويحفظ نسخة على القرص.'''
    outputs_bhwc = outputs_to_bhwc(outputs)
    image_tensor = images[sample_index].detach().float().cpu()

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image = (image_tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    height, width = image.shape[:2]

    probabilities = torch.sigmoid(outputs_bhwc[sample_index].float()).cpu().numpy()
    region_map = cv2.resize(probabilities[:, :, 0], (width, height), interpolation=cv2.INTER_LINEAR)
    boxes = extract_prediction_boxes(region_map, threshold=threshold, min_area=min_area)

    boxed_image = np.ascontiguousarray((image * 255.0).round().astype(np.uint8))
    for index, box in enumerate(boxes, start=1):
        cv2.rectangle(boxed_image, (box["x1"], box["y1"]), (box["x2"], box["y2"]), (0, 255, 0), 2)
        cv2.putText(boxed_image, str(index), (box["x1"], max(18, box["y1"] - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2, cv2.LINE_AA)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image); axes[0].set_title("الصورة الأصلية"); axes[0].axis("off")
    axes[1].imshow(boxed_image); axes[1].set_title(f"توقع النموذج | صناديق={len(boxes)}"); axes[1].axis("off")
    axes[2].imshow(region_map, cmap="hot", vmin=0, vmax=1); axes[2].set_title(f"خريطة Region (threshold={threshold})")
    axes[2].axis("off")
    fig.suptitle(f"CRAFT — Step {step} [{tag}]", fontsize=14)
    fig.tight_layout()

    preview_path = PREVIEW_DIR / f"preview_step_{step:08d}_{tag}.png"
    tmp_path = Path(str(preview_path) + ".tmp.png")
    fig.savefig(tmp_path, dpi=110, bbox_inches="tight")
    os.replace(tmp_path, preview_path)
    plt.show()
    plt.close(fig)
    return preview_path, boxes


print("تم تعريف: verify_batch, run_validation, save_prediction_preview")


تم تعريف: verify_batch, run_validation, save_prediction_preview


## 9) حلقة التدريب الرئيسية

- كل خطوة: توليد Batch إجرائي في الذاكرة وفق مرحلة التعلّم التدريجي الحالية → forward → OHEM loss → backward.
- كل `PREVIEW_EVERY_STEPS`: معاينة بصرية (3 لوحات) تُعرض وتُحفظ.
- كل `VALIDATE_EVERY_STEPS`: تحقق كامل بأصعب صعوبة ثابتة، وحفظ أفضل نموذج تلقائياً عند التحسّن.
- كل `SAVE_EVERY_STEPS`: حفظ آخر نموذج (`craft_latest.pt`) للاستئناف الآمن لاحقاً.


In [ ]:
import copy


def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = Path(str(path) + ".tmp")
    try:
        torch.save(payload, tmp_path)
        os.replace(tmp_path, path)
    finally:
        if tmp_path.exists():
            tmp_path.unlink()
    print(f"  💾 تم الحفظ: {path.name} ({path.stat().st_size / (1024 ** 2):.2f} MB)")


def build_checkpoint(step, train_loss, val_loss, best_val_loss, best_step):
    return {
        "model": copy.deepcopy(model.state_dict()),
        "optimizer": copy.deepcopy(optimizer.state_dict()),
        "step": int(step),
        "train_loss": float(train_loss),
        "val_loss": None if val_loss is None else float(val_loss),
        "best_val_loss": float(best_val_loss),
        "best_step": int(best_step),
        "img_size": IMG_SIZE,
        "target_size": TARGET_SIZE,
    }


print("تم تعريف: atomic_torch_save, build_checkpoint")


تم تعريف: atomic_torch_save, build_checkpoint


In [ ]:
model.train()
end_step = start_step + STEPS_THIS_SESSION
print(f"بدء/استئناف التدريب من Step={start_step} إلى Step={end_step} "
      f"(من إجمالي منحنى تدريجي طوله {TOTAL_CURRICULUM_STEPS} خطوة)")

for step in range(start_step, end_step):
    batch_seed = SEED + step
    random.seed(batch_seed)
    np.random.seed(batch_seed % (2 ** 32 - 1))
    torch.manual_seed(batch_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(batch_seed)

    difficulty = curriculum_stage(step, TOTAL_CURRICULUM_STEPS)

    cpu_images, cpu_targets = generate_online_batch(BATCH_SIZE, step, IMG_SIZE, TARGET_SIZE,
                                                      font_mgr, bg_gen, corpus, difficulty)
    verify_batch(cpu_images, cpu_targets)

    images = cpu_images.to(device, non_blocking=True)
    targets = cpu_targets.to(device, non_blocking=True)
    del cpu_images, cpu_targets

    optimizer.zero_grad(set_to_none=True)
    raw_outputs, _ = model(images)
    loss = criterion(raw_outputs, targets)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Loss غير صالح عند Step={step}: {loss.item()}")

    loss_value = float(loss.detach().cpu().item())
    loss.backward()

    for parameter in model.parameters():
        if parameter.grad is not None and not torch.isfinite(parameter.grad).all():
            raise RuntimeError(f"Gradient غير صالح عند Step={step}")

    optimizer.step()
    completed_step = step + 1
    validation_loss = None

    if completed_step % PREVIEW_EVERY_STEPS == 0 or completed_step == start_step + 1:
        preview_path, predicted_boxes = save_prediction_preview(
            images.detach().cpu(), raw_outputs.detach().cpu(), completed_step,
            tag=f"train-{difficulty['name']}")
        print(f"  🖼️ معاينة محفوظة: {preview_path.name} | صناديق مكتشفة: {len(predicted_boxes)}")

    if completed_step % VALIDATE_EVERY_STEPS == 0 or completed_step == end_step:
        validation_loss = run_validation(VALIDATION_BATCHES)
        if validation_loss < best_val_loss:
            best_val_loss, best_step = validation_loss, completed_step
            best_payload = build_checkpoint(completed_step, loss_value, validation_loss, best_val_loss, best_step)
            best_payload["is_best"] = True
            atomic_torch_save(best_payload, BEST_CHECKPOINT_PATH)
            print(f"  ⭐ أفضل نموذج جديد | val_loss={best_val_loss:.6f} عند Step={best_step}")
        else:
            print(f"  (لم يتحسن val_loss: {validation_loss:.6f} مقابل الأفضل {best_val_loss:.6f})")

    if completed_step % SAVE_EVERY_STEPS == 0 or completed_step == end_step:
        latest_payload = build_checkpoint(completed_step, loss_value, validation_loss, best_val_loss, best_step)
        atomic_torch_save(latest_payload, CHECKPOINT_PATH)

    msg = f"Step {completed_step}/{end_step} [{difficulty['name']}] | Train Loss: {loss_value:.6f}"
    if validation_loss is not None:
        msg += f" | Val Loss: {validation_loss:.6f}"
    print(msg)

    del images, targets, raw_outputs, loss
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

start_step = end_step
print("\nانتهت جلسة التدريب.")
print("آخر نموذج:", CHECKPOINT_PATH)
print("أفضل نموذج:", BEST_CHECKPOINT_PATH)
print(f"أفضل Validation Loss: {best_val_loss:.6f} عند Step={best_step}")
print("لمتابعة التدريب لاحقاً: أعد تشغيل الدفتر من الأعلى (سيُستأنف تلقائياً من هذا الـ checkpoint)، "
      "وغيّر STEPS_THIS_SESSION فقط إذا رغبت في عدد خطوات مختلف.")


Step 2192/5980 [3-صعوبة متوسطة-عالية] | Train Loss: 0.008697
Step 2193/5980 [3-صعوبة متوسطة-عالية] | Train Loss: 0.008008


## 10) اختبار بصري شامل بعد التدريب (عيّنات عشوائية بمستويات صعوبة مختلفة)

يقارن الصورة الأصلية بخريطة الاحتمالية وصناديق التوقع، على عدّة عيّنات جديدة كلياً (لم يرها النموذج
أثناء التدريب) وبمستويات صعوبة مختلفة (سهل ← صعب) للتأكد من التعميم الفعلي.


In [ ]:
print("تحميل أفضل نموذج محفوظ للاختبار البصري...")
_eval_model = CRAFT(pretrained=False, freeze=False).to(device)
_ckpt_for_eval = BEST_CHECKPOINT_PATH if _valid_checkpoint(BEST_CHECKPOINT_PATH) else CHECKPOINT_PATH
if not _valid_checkpoint(_ckpt_for_eval):
    print("[تنبيه] لا يوجد أي checkpoint بعد — سيُستخدم النموذج الحالي في الذاكرة (غير مدرَّب أو مدرَّب جزئياً).")
    _eval_model.load_state_dict(model.state_dict())
else:
    _ckpt = torch.load(_ckpt_for_eval, map_location=device, weights_only=False)
    _eval_model.load_state_dict(_ckpt["model"])
    print(f"تم التحميل من: {_ckpt_for_eval.name} | Step={_ckpt.get('step')} | val_loss={_ckpt.get('val_loss')}")
_eval_model.eval()

for _tag, _difficulty in [
    ("سهل", curriculum_stage(0, TOTAL_CURRICULUM_STEPS)),
    ("متوسط", curriculum_stage(int(TOTAL_CURRICULUM_STEPS * 0.5), TOTAL_CURRICULUM_STEPS)),
    ("صعب/واقعي كامل", hardest_difficulty()),
]:
    seed = 999_000 + hash(_tag) % 1000
    random.seed(seed)
    np.random.seed(seed % (2 ** 32 - 1))
    torch.manual_seed(seed)

    x, y = generate_online_batch(1, 0, IMG_SIZE, TARGET_SIZE, font_mgr, bg_gen, corpus, _difficulty)
    with torch.no_grad():
        out, _ = _eval_model(x.to(device))
    print(f"\n=== مستوى الصعوبة: {_tag} ({_difficulty['name']}) ===")
    save_prediction_preview(x, out.detach().cpu(), step=start_step, sample_index=0,
                             threshold=0.5, min_area=3, tag=f"visual-test-{_tag}")


## 11) الاختبار النهائي: اكتشاف **كل** الحروف التايلاندية — بما فيها الحروف المهملة/الأثرية

يبني هذا الاختبار لوحة شبكية (Grid) واضحة تحتوي:
- **الحروف الساكنة الـ44 كاملة**، بما فيها الحرفان المهملان تاريخياً: **ฃ (كو-خวด)** و **ฅ (คو-คน)**.
- **الحروف/الرموز شبه الساكنة النادرة (أثرية الاستخدام)**: ฤ, ฦ, ฤๅ, ฦๅ.
- كل الحركات (สระ) والعلامات الصوتية (วรรณยุกต์) وعلامة "ทัณฑฆาต" — مرفقة بحرف قاعدة محايد للعرض الواضح.
- الأرقام التايلاندية ๐-๙، وعلامتا ฯ (ไปยาลน้อย) و ๆ (ไม้ยมก).

ثم تُمرَّر اللوحة على النموذج المدرَّب، وتُستخرج صناديق الاكتشاف، ويُقارَن كل رمز بموضعه الحقيقي المعروف
(لأن التخطيط شبكي محدَّد الإحداثيات) لإنتاج **تقرير تغطية**: كم حرفاً اكتُشف من إجمالي كل الحروف،
مع تلوين كل خانة: أخضر = تم اكتشافها، أحمر = لم تُكتشف.


In [ ]:
# التصنيفات الكاملة للأبجدية التايلاندية (بما فيها الحروف المهملة/الأثرية)
THAI_CONSONANTS_ALL = list("กขฃคฅฆงจฉชซฌญฎฏฐฑฒณดตถทธนบปผฝพฟภมยรลวศษสหฬอฮ")
THAI_OBSOLETE_CONSONANTS = ["ฃ", "ฅ"]  # كو-خวด وكو-كن: مهملتان رسمياً منذ عقود (حروف أثرية)
THAI_RARE_VOCALIC = ["ฤ", "ฦ", "ฤๅ", "ฦๅ"]  # نادرة جداً/أثرية الاستخدام (قروض سنسكريتية)
THAI_VOWEL_SIGNS = ["ะ", "ั", "า", "ำ", "ิ", "ี", "ึ", "ื", "ุ", "ู", "เ", "แ", "โ", "ใ", "ไ"]
THAI_TONE_AND_MARKS = ["่", "้", "๊", "๋", "์", "ๆ", "ฯ"]
THAI_DIGITS = list("๐๑๒๓๔๕๖๗๘๙")

# الحركات/العلامات الصوتية لا تُعرض منفردة (رموز عائمة) — تُلصَق بحرف قاعدة محايد "อ" مثل معاجم الخط التايلاندي
_BASE_FOR_MARKS = "อ"


def _display_unit(ch: str) -> str:
    if ch in THAI_VOWEL_SIGNS or ch in THAI_TONE_AND_MARKS:
        if ch in ("เ", "แ", "โ", "ใ", "ไ"):  # حركات تُكتب قبل الحرف الأساسي
            return ch + _BASE_FOR_MARKS
        return _BASE_FOR_MARKS + ch
    return ch


FULL_TEST_UNITS = []  # كل عنصر: (النص المعروض, تسمية توضيحية, هل هو "أثري/مهمل")
for c in THAI_CONSONANTS_ALL:
    FULL_TEST_UNITS.append((c, c, c in THAI_OBSOLETE_CONSONANTS))
for c in THAI_RARE_VOCALIC:
    FULL_TEST_UNITS.append((c, c, True))
for c in THAI_VOWEL_SIGNS:
    FULL_TEST_UNITS.append((_display_unit(c), c, False))
for c in THAI_TONE_AND_MARKS:
    FULL_TEST_UNITS.append((_display_unit(c), c, False))
for c in THAI_DIGITS:
    FULL_TEST_UNITS.append((c, c, False))

print(f"إجمالي عدد الوحدات في اختبار التغطية الكامل: {len(FULL_TEST_UNITS)}")
print(f"منها حروف مهملة/أثرية صريحة: {sum(1 for _, _, a in FULL_TEST_UNITS if a)} "
      f"({', '.join(THAI_OBSOLETE_CONSONANTS + THAI_RARE_VOCALIC)})")


In [ ]:
def build_full_charset_grid_image(font_mgr, units, cell_size=96, cols=12, font_size=48, busy_bg=False):
    '''يبني صورة شبكية بها كل رمز في خانة بمكان معروف مسبقاً (ground truth دقيق)،
    ويرجع الصورة + قائمة (النص, x_center, y_center, x1,y1,x2,y2, هل_أثري).'''
    rows = math.ceil(len(units) / cols)
    width = cols * cell_size
    height = rows * cell_size

    if busy_bg:
        canvas_bg = bg_gen.generate(complexity=0.5).resize((width, height)).convert("RGB")
    else:
        canvas_bg = Image.new("RGB", (width, height), (250, 250, 248))

    draw = ImageDraw.Draw(canvas_bg)
    font = font_mgr.font_at(0, font_size)

    ground_truth = []
    for idx, (display_text, label, is_archaic) in enumerate(units):
        row, col = divmod(idx, cols)
        x1, y1 = col * cell_size, row * cell_size
        x2, y2 = x1 + cell_size, y1 + cell_size
        # إطار خفيف لتمييز الخلايا بصرياً فقط (لا يؤثر على GT)
        draw.rectangle([x1 + 1, y1 + 1, x2 - 2, y2 - 2], outline=(210, 210, 205), width=1)
        try:
            bbox = draw.textbbox((0, 0), display_text, font=font)
            tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
            tx = x1 + (cell_size - tw) / 2 - bbox[0]
            ty = y1 + (cell_size - th) / 2 - bbox[1] - int(cell_size * 0.08)
        except Exception:
            tx, ty = x1 + cell_size * 0.2, y1 + cell_size * 0.2
        draw.text((tx, ty), display_text, font=font, fill=(15, 15, 15))
        draw.text((x1 + 4, y2 - 16), label, font=font_mgr.font_at(1, 12), fill=(120, 120, 120))

        cx, cy = (x1 + x2) / 2, (y1 + y2 - int(cell_size * 0.12)) / 2 + y1 * 0
        cx, cy = x1 + cell_size / 2, y1 + cell_size / 2 - cell_size * 0.06
        ground_truth.append({
            "label": label, "is_archaic": is_archaic,
            "cx": cx, "cy": cy, "x1": x1, "y1": y1, "x2": x2, "y2": y2,
        })

    return canvas_bg, ground_truth


print("تم تعريف: build_full_charset_grid_image")


In [ ]:
# بناء لوحة الاختبار الشاملة وتشغيل النموذج المدرَّب عليها
test_image, ground_truth = build_full_charset_grid_image(font_mgr, FULL_TEST_UNITS, cell_size=96, cols=12,
                                                           font_size=46, busy_bg=False)
print("أبعاد صورة اختبار كل الحروف:", test_image.size, "| عدد الوحدات:", len(ground_truth))

img_np = np.asarray(test_image.convert("RGB"))
img_res, ratio, _ = resize_aspect_ratio(img_np, max(test_image.size) + (32 - max(test_image.size) % 32) % 32,
                                         interpolation=cv2.INTER_LINEAR)
input_tensor = torch.from_numpy(normalizeMeanVariance(img_res)).permute(2, 0, 1).float().unsqueeze(0).to(device)

with torch.no_grad():
    raw_out, _ = _eval_model(input_tensor)

probabilities = torch.sigmoid(outputs_to_bhwc(raw_out)[0].float()).cpu().numpy()
region_map = cv2.resize(probabilities[:, :, 0], test_image.size, interpolation=cv2.INTER_LINEAR)

# عتبة اكتشاف منخفضة نسبياً هنا: الهدف قياس التغطية الأقصى الممكنة للنموذج على كل الرموز
detected_boxes = extract_prediction_boxes(region_map, threshold=0.35, min_area=4)
print("عدد صناديق الاكتشاف الخام من النموذج:", len(detected_boxes))


In [ ]:
# مطابقة كل رمز حقيقي بأقرب صندوق اكتشاف (تداخل مركز الصندوق مع خانة الرمز)
def box_center(b):
    return (b["x1"] + b["x2"]) / 2.0, (b["y1"] + b["y2"]) / 2.0


detected_flags = []
for gt in ground_truth:
    hit = False
    for b in detected_boxes:
        bx, by = box_center(b)
        if gt["x1"] <= bx <= gt["x2"] and gt["y1"] <= by <= gt["y2"]:
            hit = True
            break
        cx, cy = (gt["x1"] + gt["x2"]) / 2, (gt["y1"] + gt["y2"]) / 2
        if b["x1"] <= cx <= b["x2"] and b["y1"] <= cy <= b["y2"]:
            hit = True
            break
    detected_flags.append(hit)

total = len(ground_truth)
total_detected = sum(detected_flags)
archaic_items = [(g, d) for g, d in zip(ground_truth, detected_flags) if g["is_archaic"]]
archaic_detected = sum(1 for _, d in archaic_items if d)

print("=" * 60)
print(f"التغطية الإجمالية: {total_detected}/{total} رمزاً تم اكتشافه "
      f"({100.0 * total_detected / max(1, total):.1f}%)")
print(f"تغطية الحروف الأثرية/المهملة تحديداً: {archaic_detected}/{len(archaic_items)} "
      f"({', '.join(g['label'] for g, d in archaic_items)})")
print("=" * 60)

missed = [g["label"] for g, d in zip(ground_truth, detected_flags) if not d]
if missed:
    print("الرموز التي لم يكتشفها النموذج بعد:", " ".join(missed))
else:
    print("تم اكتشاف كل الرموز بنجاح، بما فيها كل الحروف الأثرية/المهملة.")

# رسم اللوحة النهائية: أخضر = مكتشف، أحمر = غير مكتشف، إطار أصفر = حرف أثري
vis = np.array(test_image.convert("RGB")).copy()
for gt, ok in zip(ground_truth, detected_flags):
    color = (0, 200, 0) if ok else (220, 30, 30)
    cv2.rectangle(vis, (gt["x1"] + 2, gt["y1"] + 2), (gt["x2"] - 2, gt["y2"] - 2), color, 2)
    if gt["is_archaic"]:
        cv2.rectangle(vis, (gt["x1"], gt["y1"]), (gt["x2"], gt["y2"]), (255, 210, 0), 2)

fig, axes = plt.subplots(1, 2, figsize=(20, max(6, 20 * vis.shape[0] / vis.shape[1])))
axes[0].imshow(vis)
axes[0].set_title(f"تقرير التغطية: {total_detected}/{total} — إطار أصفر = حرف أثري/مهمل")
axes[0].axis("off")
axes[1].imshow(region_map, cmap="hot", vmin=0, vmax=1)
axes[1].set_title("خريطة Region الكاملة على لوحة الاختبار")
axes[1].axis("off")
fig.tight_layout()

_final_report_path = PREVIEW_DIR / "full_thai_charset_coverage_report.png"
fig.savefig(_final_report_path, dpi=130, bbox_inches="tight")
plt.show()
plt.close(fig)
print("تم حفظ تقرير التغطية النهائي في:", _final_report_path)


## ملاحظات الاستئناف بين الجلسات/الأيام

- أعد تشغيل الدفتر من الأعلى بالترتيب. سيكتشف تلقائياً `craft_latest.pt` إن وُجد ويكمل منه (`start_step` يُقرأ من الملف).
- **جديد**: إن لم يوجد `craft_latest.pt` بعد (أول تشغيل)، سيبدأ التدريب من أوزان CRAFT الرسمية المُدرَّبة مسبقاً تلقائياً (تخصيص) بدلاً من أوزان عشوائية، طالما أن `craft_mlt_25k.pth` موجود في `PRETRAINED_CRAFT_PATH` (تُنزَّل تلقائياً في خلية مبكرة، أو ضعه يدوياً إن فشل التنزيل).
- غيّر فقط `STEPS_THIS_SESSION` في خلية الإعدادات الفائقة لكل جلسة جديدة.
- منحنى التعلّم التدريجي محسوب دائماً كنسبة `step / TOTAL_CURRICULUM_STEPS`، لذا استمرار نفس القيمة لـ `TOTAL_CURRICULUM_STEPS` عبر الجلسات ضروري لثبات تدرّج الصعوبة.
- لا تُشغِّل خلية التدريب أكثر من مرة داخل نفس الجلسة قبل مراجعة `start_step` — كل تشغيل يبدأ من `start_step` الحالي حتى `start_step + STEPS_THIS_SESSION`.
